In [3]:
import duckdb
import time

# Connect to a DuckDB database file
# This creates a persistent file we can reopen later instead of re-running everything
con = duckdb.connect("../data/spotify.duckdb")

# Count the rows in the CSV directly, no need to load it into memory
# DuckDB scans the file efficiently
start = time.time()

result = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto('../data/charts.csv')
""").fetchone()

elapsed = time.time() - start
print(f"Total rows: {result[0]:,}")
print(f"Time: {elapsed:.1f} seconds")

Total rows: 26,173,514
Time: 0.9 seconds


In [4]:
# Import the CSV into a real DuckDB table named "charts"
# After this, queries run against the table directly, much faster than re-reading the CSV
start = time.time()

con.execute("""
    CREATE OR REPLACE TABLE charts AS
    SELECT * FROM read_csv_auto('../data/charts.csv')
""")

elapsed = time.time() - start
print(f"Import time: {elapsed:.1f} seconds")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Import time: 7.8 seconds


In [3]:
# What columns do we have?
print(con.execute("DESCRIBE charts").df())

  column_name column_type null   key default extra
0       title     VARCHAR  YES  None    None  None
1        rank      BIGINT  YES  None    None  None
2        date        DATE  YES  None    None  None
3      artist     VARCHAR  YES  None    None  None
4         url     VARCHAR  YES  None    None  None
5      region     VARCHAR  YES  None    None  None
6       chart     VARCHAR  YES  None    None  None
7       trend     VARCHAR  YES  None    None  None
8     streams      BIGINT  YES  None    None  None


In [4]:
# Sample some rows to see what the data actually looks like
print(con.execute("SELECT * FROM charts LIMIT 5").df())

                         title  rank       date  \
0      Chantaje (feat. Maluma)     1 2017-01-01   
1  Vente Pa' Ca (feat. Maluma)     2 2017-01-01   
2   Reggaetón Lento (Bailemos)     3 2017-01-01   
3                       Safari     4 2017-01-01   
4                  Shaky Shaky     5 2017-01-01   

                                  artist  \
0                                Shakira   
1                           Ricky Martin   
2                                   CNCO   
3  J Balvin, Pharrell Williams, BIA, Sky   
4                           Daddy Yankee   

                                                 url     region   chart  \
0  https://open.spotify.com/track/6mICuAdrwEjh6Y6...  Argentina  top200   
1  https://open.spotify.com/track/7DM4BPaS7uofFul...  Argentina  top200   
2  https://open.spotify.com/track/3AEZUABDXNtecAO...  Argentina  top200   
3  https://open.spotify.com/track/6rQSrBHf7HlZjtc...  Argentina  top200   
4  https://open.spotify.com/track/58IL315gMSTD37D... 

In [5]:
# How many distinct countries (regions) are in the data?
# How many days does the data cover?
result = con.execute("""
    SELECT 
        COUNT(DISTINCT region) AS countries,
        MIN(date) AS earliest,
        MAX(date) AS latest,
        COUNT(DISTINCT date) AS days_covered
    FROM charts
""").df()
print(result)

   countries   earliest     latest  days_covered
0         70 2017-01-01 2021-12-31          1826


In [10]:
# Run Question 1: market concentration by country
result = con.execute(open("../sql/01_market_concentration.sql").read()).df()
print(result.head(15))
print("...")
print(result.tail(15))

           region  top10_appearances  total_appearances  pct_from_top10
0           Japan            89272.0           356774.0           25.02
1         Iceland            44812.0           212123.0           21.13
2       Singapore            74142.0           358388.0           20.69
3     New Zealand            71013.0           358387.0           19.81
4   United States            71733.0           364184.0           19.70
5     Philippines            65540.0           358390.0           18.29
6       Hong Kong            65371.0           358388.0           18.24
7         Estonia            19231.0           108007.0           17.81
8          Brazil            64571.0           364516.0           17.71
9        Malaysia            61984.0           358390.0           17.30
10         France            61933.0           358390.0           17.28
11         Canada            62316.0           361390.0           17.24
12       Honduras            48664.0           286269.0         

In [11]:
# Save the result so we can use it later for charts and the dashboard
result.to_csv("../data/01_market_concentration.csv", index=False)
print("Saved to data/01_market_concentration.csv")

Saved to data/01_market_concentration.csv


In [8]:
# Check which countries have too little data to compare fairly
result_with_days = con.execute("""
    SELECT 
        region,
        COUNT(DISTINCT date) AS days_with_data
    FROM charts
    WHERE chart = 'top200'
    GROUP BY region
    ORDER BY days_with_data ASC
    LIMIT 15
""").df()
print(result_with_days)

                  region  days_with_data
0             Luxembourg             297
1            South Korea             302
2                 Russia             504
3                Ukraine             504
4                  Egypt             787
5                Morocco             866
6           Saudi Arabia             912
7   United Arab Emirates             932
8                  India            1008
9              Nicaragua            1100
10              Bulgaria            1157
11          South Africa            1280
12               Romania            1358
13                Israel            1358
14               Vietnam            1358


In [5]:
# Question 2: Song lifecycle for #1 hits in the United States
result = con.execute(open("../sql/02_song_lifecycle.sql").read()).df()
print(f"Total songs that hit #1 in the US: {len(result)}")
print()
print(result.head(15))

Total songs that hit #1 in the US: 138

                                            title  \
0             Bad and Boujee (feat. Lil Uzi Vert)   
1                                    Shape of You   
2                                    Passionfruit   
3                                         HUMBLE.   
4              Despacito (Featuring Daddy Yankee)   
5                               Despacito - Remix   
6                                   XO TOUR Llif3   
7   Wild Thoughts (feat. Rihanna & Bryson Tiller)   
8                                   Unforgettable   
9                                    Bank Account   
10                       Look What You Made Me Do   
11                               ...Ready For It?   
12                                 1-800-273-8255   
13                                       rockstar   
14                All I Want for Christmas Is You   

                                     artist first_chart_date  \
0                                     Migos    

In [8]:
# Aggregate: average lifecycle stats by era
era_summary = con.execute("""
    WITH lifecycle AS (
        SELECT * FROM (""" + open("../sql/02_song_lifecycle.sql").read().rstrip(";") + """)
    )
    SELECT
        era,
        COUNT(*) AS num_hits,
        ROUND(AVG(days_to_reach_number_one), 1) AS avg_days_to_climb,
        ROUND(AVG(days_at_number_one), 1) AS avg_days_at_top,
        ROUND(AVG(total_days_on_chart), 1) AS avg_total_days_on_chart,
        ROUND(AVG(days_after_peak), 1) AS avg_days_after_peak
    FROM lifecycle
    GROUP BY era
    ORDER BY 
        CASE era
            WHEN 'Early (2017-18)' THEN 1
            WHEN 'Mid (2019-mid 2020)' THEN 2
            WHEN 'Late (mid 2020-21)' THEN 3
        END
""").df()

print(era_summary)

                   era  num_hits  avg_days_to_climb  avg_days_at_top  \
0      Early (2017-18)        43               22.6             17.7   
1  Mid (2019-mid 2020)        42                5.7             12.7   
2   Late (mid 2020-21)        49                5.7             10.1   

   avg_total_days_on_chart  avg_days_after_peak  
0                    452.7                640.2  
1                    331.6                393.6  
2                    199.6                218.9  


In [7]:
# Check the songs with the longest "days to climb" in the late era
diagnostic = con.execute(open("../sql/02_song_lifecycle.sql").read()).df()

late_era = diagnostic[diagnostic['era'] == 'Late (mid 2020-21)'].copy()
late_era_long = late_era.sort_values('days_to_reach_number_one', ascending=False).head(10)
print(late_era_long[['title', 'artist', 'days_to_reach_number_one', 'first_chart_date', 'first_number_one_date']])

                                         title  \
134                            White Christmas   
129                               Monster Mash   
103          Rockin' Around The Christmas Tree   
137                                 Heat Waves   
104                     Mood (feat. iann dior)   
96   Lemonade (feat. Gunna, Don Toliver & NAV)   
101                                     DÁKITI   
115                   Kiss Me More (feat. SZA)   
106                                  Good Days   
132                      Smokin Out The Window   

                                                artist  \
134  Bing Crosby, Ken Darby Singers, John Scott Tro...   
129           Bobby "Boris" Pickett, The Crypt-Kickers   
103                                         Brenda Lee   
137                                      Glass Animals   
104                                           24kGoldn   
96                                      Internet Money   
101                             Bad Bunny, 

In [9]:
# Save the lifecycle results (filtered, no catalog songs)
result_clean = con.execute(open("../sql/02_song_lifecycle.sql").read()).df()
result_clean.to_csv("../data/02_song_lifecycle.csv", index=False)
era_summary.to_csv("../data/02_song_lifecycle_by_era.csv", index=False)
print(f"Saved {len(result_clean)} hits to data/02_song_lifecycle.csv")
print(f"Saved era summary to data/02_song_lifecycle_by_era.csv")

Saved 134 hits to data/02_song_lifecycle.csv
Saved era summary to data/02_song_lifecycle_by_era.csv


In [10]:
# Question 3: Single-artist dominance per country
result = con.execute(open("../sql/03_artist_dominance.sql").read()).df()
print(f"Number of countries analyzed: {len(result)}")
print()
print(result)

Number of countries analyzed: 54

                region             top_artist  top_artist_appearances  \
0             Honduras              Bad Bunny                   16182   
1   Dominican Republic              Bad Bunny                   17773   
2          El Salvador              Bad Bunny                   13564   
3          New Zealand                  SIX60                   14405   
4              Iceland            Hafdís Huld                    8324   
5               Panama              Bad Bunny                   12796   
6            Australia             Ed Sheeran                   13194   
7            Singapore             Ed Sheeran                   12967   
8               France                  Ninho                   12785   
9            Guatemala              Bad Bunny                   12544   
10               Japan  Official HIGE DANdism                   11866   
11               Chile              Bad Bunny                   11926   
12      United Ki

In [11]:
# Save Q3 result
result.to_csv("../data/03_artist_dominance.csv", index=False)
print(f"Saved {len(result)} countries to data/03_artist_dominance.csv")

Saved 54 countries to data/03_artist_dominance.csv
